<a href="https://colab.research.google.com/github/risinukaviranga/ci-cd-build-failure-prediction/blob/main/Feature_engineering_and_feature_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q duckdb pandas numpy pyarrow scikit-learn matplotlib seaborn

In [2]:
import duckdb
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

INPUT_PATH = "/content/drive/MyDrive/PipelineX/data/processed/cleaned_dataset.parquet"

OUTPUT_PATH = "/content/drive/MyDrive/PipelineX/data/processed"

REPORT_PATH = "/content/drive/MyDrive/PipelineX/reports/feature_engineering"

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

con = duckdb.connect("/content/pipelinex.duckdb")

con.execute("SET memory_limit='4GB'")
con.execute("SET threads=2")
con.execute("SET temp_directory='/content/duckdb_temp'")

parquet_path = INPUT_PATH.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW cleaned AS
    SELECT *
    FROM read_parquet('{parquet_path}')
""")

print("Cleaned dataset loaded successfully!")

Mounted at /content/drive
Cleaned dataset loaded successfully!


In [3]:
print("Total records:",
      con.execute("SELECT COUNT(*) FROM cleaned").fetchone()[0])

print("Total columns:",
      len(con.execute("DESCRIBE SELECT * FROM cleaned").df()))

display(con.execute("DESCRIBE SELECT * FROM cleaned").df())

Total records: 3841797
Total columns: 67


,column_name,column_type,null,key,default,extra
0,tr_build_id,BIGINT,YES,None,None,None
1,gh_project_name,VARCHAR,YES,None,None,None
2,gh_is_pr,BOOLEAN,YES,None,None,None
3,gh_pr_created_at,VARCHAR,YES,None,None,None
4,gh_pull_req_num,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
62,tr_original_commit,VARCHAR,YES,None,None,None
63,tr_duration,VARCHAR,YES,None,None,None
64,tr_status,VARCHAR,YES,None,None,None
65,tr_jobs,VARCHAR,YES,None,None,None


Review available features

In [4]:
schema = con.execute("DESCRIBE SELECT * FROM cleaned").df()

available_columns = schema["column_name"].tolist()

print("Available columns:")

for column in available_columns:
    print(column)

Available columns:
tr_build_id
gh_project_name
gh_is_pr
gh_pr_created_at
gh_pull_req_num
gh_lang
git_merged_with
git_branch
gh_num_commits_in_push
gh_commits_in_push
git_prev_commit_resolution_status
git_prev_built_commit
tr_prev_build
gh_first_commit_created_at
gh_team_size
git_all_built_commits
git_num_all_built_commits
git_trigger_commit
tr_virtual_merged_into
gh_num_issue_comments
gh_num_commit_comments
gh_num_pr_comments
git_diff_src_churn
git_diff_test_churn
gh_diff_files_added
gh_diff_files_deleted
gh_diff_files_modified
gh_diff_tests_added
gh_diff_tests_deleted
gh_diff_src_files
gh_diff_doc_files
gh_diff_other_files
gh_num_commits_on_files_touched
gh_sloc
gh_test_lines_per_kloc
gh_test_cases_per_kloc
gh_asserts_cases_per_kloc
gh_by_core_team_member
gh_description_complexity
gh_pushed_at
gh_build_started_at
gh_repo_age
gh_repo_num_commits
tr_job_id
tr_build_number
tr_log_lan
tr_log_status
tr_log_setup_time
tr_log_analyzer
tr_log_frameworks
tr_log_bool_tests_ran
tr_log_bool_tests

In [5]:
candidate_columns = [
    "tr_build_id",
    "tr_job_id",
    "build_failure",
    "tr_status",
    "gh_is_pr",
    "gh_lang",
    "git_branch",
    "gh_num_commits_in_push",
    "gh_commits_in_push",
    "gh_build_started_at",
    "gh_project_name"
]

for column in candidate_columns:
    if column in available_columns:
        print("Available:", column)
    else:
        print("Not available:", column)

Available: tr_build_id
Available: tr_job_id
Available: build_failure
Available: tr_status
Available: gh_is_pr
Available: gh_lang
Available: git_branch
Available: gh_num_commits_in_push
Available: gh_commits_in_push
Available: gh_build_started_at
Available: gh_project_name


Identify and exclude data leakage

In [6]:
excluded_columns = [
    "tr_build_id",
    "tr_job_id",
    "tr_status",
    "build_failure",
    "tr_log_status"
]

print("Initial excluded columns:")

for column in excluded_columns:
    if column in available_columns:
        print(column)

Initial excluded columns:
tr_build_id
tr_job_id
tr_status
build_failure
tr_log_status


In [7]:
feature_audit = pd.DataFrame({
    "column_name": available_columns
})

feature_audit["decision"] = "Needs review"
feature_audit["reason"] = "Check meaning and prediction-time availability"

feature_audit.loc[
    feature_audit["column_name"].isin(excluded_columns),
    "decision"
] = "Exclude"

display(feature_audit)

,column_name,decision,reason
0,tr_build_id,Exclude,Check meaning and prediction-time availability
1,gh_project_name,Needs review,Check meaning and prediction-time availability
2,gh_is_pr,Needs review,Check meaning and prediction-time availability
3,gh_pr_created_at,Needs review,Check meaning and prediction-time availability
4,gh_pull_req_num,Needs review,Check meaning and prediction-time availability
...,...,...,...
62,tr_original_commit,Needs review,Check meaning and prediction-time availability
63,tr_duration,Needs review,Check meaning and prediction-time availability
64,tr_status,Exclude,Check meaning and prediction-time availability
65,tr_jobs,Needs review,Check meaning and prediction-time availability


Create time based features

In [8]:
if "gh_build_started_at" in available_columns:

    timestamp_check = con.execute("""
        SELECT
            COUNT(*) AS total_records,
            COUNT(
                TRY_CAST(gh_build_started_at AS TIMESTAMP)
            ) AS valid_timestamps
        FROM cleaned
    """).df()

    display(timestamp_check)

else:
    print("Timestamp column not found.")

,total_records,valid_timestamps
0,3841797,3841797


In [9]:
if "gh_build_started_at" in available_columns:

    con.execute("""
        CREATE OR REPLACE TABLE feature_data AS

        SELECT
            *,

            EXTRACT(
                YEAR FROM TRY_CAST(gh_build_started_at AS TIMESTAMP)
            ) AS build_year,

            EXTRACT(
                MONTH FROM TRY_CAST(gh_build_started_at AS TIMESTAMP)
            ) AS build_month,

            EXTRACT(
                DAY FROM TRY_CAST(gh_build_started_at AS TIMESTAMP)
            ) AS build_day,

            EXTRACT(
                DAYOFWEEK FROM TRY_CAST(gh_build_started_at AS TIMESTAMP)
            ) AS build_day_of_week,

            EXTRACT(
                HOUR FROM TRY_CAST(gh_build_started_at AS TIMESTAMP)
            ) AS build_hour,

            CASE
                WHEN EXTRACT(
                    DAYOFWEEK FROM TRY_CAST(gh_build_started_at AS TIMESTAMP)
                ) IN (0, 6)
                THEN 1
                ELSE 0
            END AS is_weekend

        FROM cleaned
    """)

    print("Time-based features created!")

else:
    print("Review your timestamp column before creating time features.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Time-based features created!


Create commit and repository features

In [10]:
for column in [
    "gh_num_commits_in_push",
    "gh_project_name",
    "gh_lang",
    "gh_is_pr",
    "git_branch"
]:

    if column in available_columns:

        print(f"\n--- {column} ---")

        display(
            con.execute(f"""
                SELECT "{column}", COUNT(*) AS frequency
                FROM cleaned
                GROUP BY "{column}"
                ORDER BY frequency DESC
                LIMIT 10
            """).df()
        )


--- gh_num_commits_in_push ---


,gh_num_commits_in_push,frequency
0,NA,3841797



--- gh_project_name ---


,gh_project_name,frequency
0,DataDog/dd-agent,141949
1,pyca/cryptography,109868
2,opf/openproject,108663
3,bundler/bundler,80858
4,getsentry/sentry,57046
5,rollbar/rollbar-gem,48100
6,checkstyle/checkstyle,45690
7,rspec/rspec-rails,44885
8,otwcode/otwarchive,44885
9,rg3/youtube-dl,36955



--- gh_lang ---


,gh_lang,frequency
0,ruby,1685479
1,python,1502831
2,java,399083
3,go,254404



--- gh_is_pr ---


,gh_is_pr,frequency
0,False,3030009
1,True,811788



--- git_branch ---


,git_branch,frequency
0,master,2679895
1,develop,135875
2,dev,89783
3,trunk,46134
4,devel,44136
5,release/5.0,17733
6,stable,16738
7,auto,15745
8,feature/rails3,7388
9,2-99-maintenance,5344


In [11]:
if "gh_num_commits_in_push" in available_columns:

    con.execute("""
        CREATE OR REPLACE TABLE feature_data AS

        SELECT
            *,

            TRY_CAST(
                gh_num_commits_in_push AS DOUBLE
            ) AS commits_in_push_numeric

        FROM cleaned
    """)

    print("Commit count feature created!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Commit count feature created!


In [12]:
if "gh_num_commits_in_push" in available_columns:

    con.execute("""
        CREATE OR REPLACE TABLE feature_data AS

        SELECT
            *,

            TRY_CAST(
                gh_num_commits_in_push AS DOUBLE
            ) AS commits_in_push_numeric

        FROM feature_data
    """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clean categorical features

In [13]:
categorical_candidates = [
    "gh_lang",
    "git_branch",
    "gh_project_name"
]

existing_categoricals = [
    c for c in categorical_candidates
    if c in available_columns
]

print("Categorical features:", existing_categoricals)

Categorical features: ['gh_lang', 'git_branch', 'gh_project_name']


In [14]:
for column in existing_categoricals:

    new_column = column + "_clean"

    con.execute(f"""
        ALTER TABLE feature_data
        ADD COLUMN IF NOT EXISTS "{new_column}" VARCHAR
    """)

    con.execute(f"""
        UPDATE feature_data
        SET "{new_column}" =
            NULLIF(
                LOWER(TRIM(CAST("{column}" AS VARCHAR))),
                ''
            )
    """)

print("Categorical standardization completed!")

Categorical standardization completed!


Analyse feature quality

A.Missing value analysis

In [15]:
feature_schema = con.execute(
    "DESCRIBE SELECT * FROM feature_data"
).df()

feature_columns = feature_schema["column_name"].tolist()

missing_queries = []

for column in feature_columns:

    safe_column = '"' + column.replace('"', '""') + '"'

    missing_queries.append(f"""
        SELECT
            '{column}' AS feature,
            SUM(
                CASE
                    WHEN {safe_column} IS NULL THEN 1
                    ELSE 0
                END
            ) AS missing_count
        FROM feature_data
    """)

missing_sql = " UNION ALL ".join(missing_queries)

feature_missing_report = con.execute(missing_sql).df()

feature_total = con.execute(
    "SELECT COUNT(*) FROM feature_data"
).fetchone()[0]

feature_missing_report["missing_percentage"] = (
    feature_missing_report["missing_count"]
    / feature_total * 100
)

feature_missing_report = feature_missing_report.sort_values(
    "missing_percentage",
    ascending=False
)

display(feature_missing_report.head(30))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,feature,missing_count,missing_percentage
67,commits_in_push_numeric,3841797.0,100.000000
68,commits_in_push_numeric_1,3841797.0,100.000000
59,tr_log_tests_failed,3685556.0,95.933127
6,git_merged_with,3030009.0,78.869576
18,tr_virtual_merged_into,3030009.0,78.869576
49,tr_log_frameworks,1511258.0,39.337268
55,tr_log_num_tests_skipped,1445163.0,37.616850
52,tr_log_num_tests_ok,1445163.0,37.616850
53,tr_log_num_tests_failed,1445163.0,37.616850
54,tr_log_num_tests_run,1445163.0,37.616850


B.Categorical feature analysis

In [16]:
for column in existing_categoricals:

    print(f"\nUnique values in {column}:")

    result = con.execute(f"""
        SELECT
            COUNT(DISTINCT "{column}") AS unique_values
        FROM feature_data
    """).df()

    display(result)


Unique values in gh_lang:


,unique_values
0,4



Unique values in git_branch:


,unique_values
0,62687



Unique values in gh_project_name:


,unique_values
0,948


C.Numerical feature analysis

In [17]:
numeric_candidates = [
    "commits_in_push_numeric",
    "build_year",
    "build_month",
    "build_day_of_week",
    "build_hour",
    "is_weekend"
]

existing_numeric = [
    c for c in numeric_candidates
    if c in feature_columns
]

if existing_numeric:

    numeric_sql = ", ".join(
        f'"{c}"' for c in existing_numeric
    )

    numerical_report = con.execute(
        f"SUMMARIZE SELECT {numeric_sql} FROM feature_data"
    ).df()

    display(numerical_report)

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,commits_in_push_numeric,DOUBLE,None,None,0,None,None,None,None,None,3841797,100.0


Feature selection

In [18]:
approved_features = [
    "gh_is_pr",
    "gh_lang_clean",
    "git_branch_clean",
    "commits_in_push_numeric",
    "build_month",
    "build_day_of_week",
    "build_hour",
    "is_weekend"
]

approved_features = [
    c for c in approved_features
    if c in feature_columns
]

print("Approved candidate features:")

for feature in approved_features:
    print(feature)

Approved candidate features:
gh_is_pr
gh_lang_clean
git_branch_clean
commits_in_push_numeric


Create the final dataset

In [19]:
assert "build_failure" in feature_columns, \
    "build_failure column is missing!"

assert len(approved_features) > 0, \
    "No approved features found!"

In [20]:
final_columns = approved_features + ["build_failure"]

if "gh_build_started_at" in feature_columns:
    final_columns.append("gh_build_started_at")

final_columns = list(dict.fromkeys(final_columns))

print("Final dataset columns:")
print(final_columns)

Final dataset columns:
['gh_is_pr', 'gh_lang_clean', 'git_branch_clean', 'commits_in_push_numeric', 'build_failure', 'gh_build_started_at']


In [21]:
select_columns = ", ".join(
    f'"{c}"' for c in final_columns
)

con.execute(f"""
    CREATE OR REPLACE TABLE final_features AS

    SELECT
        {select_columns}

    FROM feature_data

    WHERE build_failure IS NOT NULL
""")

print("Final feature dataset created!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final feature dataset created!


In [22]:
final_summary = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT build_failure) AS target_classes,
        SUM(
            CASE WHEN build_failure = 0 THEN 1 ELSE 0 END
        ) AS successful_builds,
        SUM(
            CASE WHEN build_failure = 1 THEN 1 ELSE 0 END
        ) AS failed_builds
    FROM final_features
""").df()

display(final_summary)

,total_records,target_classes,successful_builds,failed_builds
0,3841797,2,2810783.0,1031014.0


In [23]:
display(
    con.execute("DESCRIBE SELECT * FROM final_features").df()
)

,column_name,column_type,null,key,default,extra
0,gh_is_pr,BOOLEAN,YES,None,None,None
1,gh_lang_clean,VARCHAR,YES,None,None,None
2,git_branch_clean,VARCHAR,YES,None,None,None
3,commits_in_push_numeric,DOUBLE,YES,None,None,None
4,build_failure,INTEGER,YES,None,None,None
5,gh_build_started_at,TIMESTAMP,YES,None,None,None


Export the final dataset and reports

In [24]:
FINAL_DATASET_PATH = (
    OUTPUT_PATH + "/feature_engineered_dataset.parquet"
)

final_path = FINAL_DATASET_PATH.replace("'", "''")

con.execute(f"""
    COPY final_features
    TO '{final_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Final feature dataset exported!")

Final feature dataset exported!


In [25]:
feature_audit.to_csv(
    f"{REPORT_PATH}/feature_audit.csv",
    index=False
)

feature_missing_report.to_csv(
    f"{REPORT_PATH}/feature_missing_report.csv",
    index=False
)

pd.DataFrame({
    "approved_feature": approved_features
}).to_csv(
    f"{REPORT_PATH}/approved_features.csv",
    index=False
)

final_summary.to_csv(
    f"{REPORT_PATH}/final_dataset_summary.csv",
    index=False
)

print("All feature engineering reports exported!")

All feature engineering reports exported!


# Milestone 04 – Feature Engineering Findings

## 1. Dataset Overview
- Input records:
- Input features:

## 2. Feature Engineering
- Time-based features created:
- Commit-related features created:
- Categorical features standardized:

## 3. Data Leakage Prevention
- Excluded identifiers:
- Excluded post-build features:
- Prediction-time availability checks:

## 4. Feature Quality
- Features with high missing values:
- Constant features:
- Categorical features with high cardinality:

## 5. Feature Selection
- Total approved features:
- Selected numerical features:
- Selected categorical features:
- Excluded features and reasons:

## 6. Final Dataset
- Total records:
- Total input features:
- Successful builds:
- Failed builds:
- Export format: Parquet

## 7. Conclusion
- Most promising candidate features:
- Remaining concerns:
- Next steps: